# OFL Competency Question Evaluation

This notebook evaluates the **Ontology of Flaps in Plastic and Reconstructive Surgery (OFL v2.1.0)** against its competency questions (CQ1–CQ17) and five task-based use cases.

Each CQ is answered against the **TBox** (class definitions and restrictions) and the **ABox** (synthetic flap individuals), demonstrating both ontological coverage and instance-level queryability.

| File | Role |
|------|------|
| `ofl_2.1.0.ttl` | TBox — OWL Turtle |
| `ofl_abox.owl` | ABox — synthetic individuals (non-reasoned) |
| `fma_obo.owl` | FMA — label resolution only |

**Reproduce:** install `rdflib` and `pandas`, place the three files in `ontologies/`, then run all cells from top to bottom.

In [ ]:
from rdflib import Graph, Namespace, URIRef, RDF, RDFS, OWL
from rdflib.namespace import OWL as OWL_NS
import pandas as pd
from IPython.display import display
from pathlib import Path

OFL = Namespace('https://purl.bioontology.org/ontology/OFL/')
OBO = Namespace('http://purl.obolibrary.org/obo/')

PREFIXES = """
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:     <http://www.w3.org/2002/07/owl#>
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX ofl:     <https://purl.bioontology.org/ontology/OFL/>
PREFIX dcterms: <http://purl.org/dc/terms/>
"""

ONT  = Path('..') / 'ontologies'
TBOX = ONT / 'ofl_2.1.0.ttl'   # Turtle; load with format='turtle'
ABOX = ONT / 'ofl_abox.owl'    # generated ABox (non-reasoned OWL RL)
FMA  = ONT / 'fma_obo.owl'

g = Graph()
g.parse(str(TBOX), format='turtle'); n_tbox = len(g); print(f'TBox: {n_tbox:,} triples')
g.parse(str(ABOX), format='xml');    n_abox = len(g) - n_tbox; print(f'ABox: {n_abox:,} triples')
g.parse(str(FMA),  format='xml');    print(f'FMA:  {len(g)-n_tbox-n_abox:,} triples')
print(f'Total: {len(g):,} triples')

g_tbox = Graph()
g_tbox.parse(str(TBOX), format='turtle')
g_tbox.parse(str(FMA),  format='xml')
print(f'TBox-only graph: {len(g_tbox):,} triples')

In [6]:
def run(sparql, limit=20, title="", graph=None):
    if graph is None:
        graph = g
    result = graph.query(PREFIXES + sparql)
    cols = [str(v) for v in result.vars]
    rows = list(result)
    data = [{c: (str(getattr(r, c)) if getattr(r, c) is not None else "—") for c in cols} for r in rows]
    df = pd.DataFrame(data, columns=cols)
    if title:
        print(f"\n{'='*60}")
        print(title)
    print(f"  {len(rows):,} row(s)" + (f" (showing first {limit})" if len(rows) > limit else ""))
    display(df.head(limit))
    return df

---
## Competency Questions

### CQ1 — Anatomical Composition
*What anatomical structures compose the flap?*

**Fulfilled: Yes**

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10002

Targeted axiom: 'Surgical flap' 'Ofl part' some 'Anatomical entity'

In [ ]:
run("""
SELECT DISTINCT ?flapLabel ?partLabel
WHERE {
  ?flap rdfs:subClassOf* ofl:OFLID10002 .
  {
    # hasPart restriction carried via rdfs:subClassOf
    ?flap rdfs:subClassOf ?r .
    ?r a owl:Restriction ; owl:onProperty ofl:OFLID13296 ; owl:someValuesFrom ?filler .
  } UNION {
    # hasPart restriction inside owl:equivalentClass intersection (variant classes)
    ?flap owl:equivalentClass ?eqExpr .
    ?eqExpr owl:intersectionOf ?iLst .
    ?iLst rdf:rest*/rdf:first ?r .
    ?r a owl:Restriction ; owl:onProperty ofl:OFLID13296 ; owl:someValuesFrom ?filler .
  }
  ?flap rdfs:label ?flapLabel .
  {
    # direct named filler
    FILTER(isIRI(?filler))
    OPTIONAL { ?filler rdfs:label ?partLabel }
  } UNION {
    # filler is a union — unwrap one level
    ?filler owl:unionOf ?lst .
    ?lst rdf:rest*/rdf:first ?member .
    FILTER(isIRI(?member))
    OPTIONAL { ?member rdfs:label ?partLabel }
  } UNION {
    # filler is an intersection — unwrap one level
    ?filler owl:intersectionOf ?lst .
    ?lst rdf:rest*/rdf:first ?member .
    FILTER(isIRI(?member))
    OPTIONAL { ?member rdfs:label ?partLabel }
  }
} ORDER BY ?flapLabel ?partLabel
""", title="CQ1 TBox — flap classes with compositional has-part restrictions (labels resolved)")

In [ ]:
run("""
SELECT ?componentLabel ?typeLabel
WHERE {
  ?flap rdf:type owl:NamedIndividual ;
        rdfs:label ?fl .
  FILTER(CONTAINS(LCASE(?fl), "anterolateral thigh"))
  ?flap ofl:OFLID13296 ?component .
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
} ORDER BY ?componentLabel LIMIT 10
""", title="CQ1 ABox — anatomical components of a generated ALT flap")

### CQ2 — Size
*What is the size of the flap?*

**Fulfilled: Yes** — volume and mass qualities are asserted on each flap individual via `has quality` (RO:0000086).

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10002

Targeted axiom: 'Surgical flap' 'has quality' some 'Mass'

Targeted axiom: 'Surgical flap' 'has quality' some 'Volume'

In [ ]:
run("""
SELECT ?flapLabel ?qualityLabel
WHERE {
  ?flap rdf:type ?cls .
  ?cls rdfs:subClassOf* ofl:OFLID10002 .
  ?flap rdf:type owl:NamedIndividual .
  ?flap obo:RO_0000086 ?quality .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?quality rdfs:label ?qualityLabel }
} ORDER BY ?flapLabel
""", title="CQ2 ABox — flap individuals with size qualities (volume / mass)")

### CQ3 — Vessel Connection
*To which vessels is the flap connected?*
Flaps need an anastomosis if they are free flaps which means they are anastomosed to another vessel. 

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID130000
Targeted axiom: 'Surgical flap'
 and ('has part' some 'Flap pedicle')
 and ('has part' some 'Flap pedicle vessel')
 and ('participates in' some 
    ('Complete flap operation process'
     and ('has part' some 'Flap vessel anastomosis process')))

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID120019 
Targeted axiom: 'Surgical vessel anastomosis process'
 and ('has specified output' some 'Flap pedicle vessel to recipient vessel anastomosis')
 and ('Ofl part of' some 'Complete flap operation process')

In [ ]:
run("""
SELECT DISTINCT ?flapLabel ?pedicleLabel
WHERE {
  # Named pedicle subclasses carry a part-of restriction back to their flap class
  ?pedCls rdfs:subClassOf+ ofl:OFLID106008 .
  ?pedCls rdfs:label ?pedicleLabel .
  OPTIONAL {
    ?pedCls owl:equivalentClass ?eq .
    ?eq owl:intersectionOf ?lst .
    ?lst rdf:rest*/rdf:first ?r .
    ?r a owl:Restriction ;
       owl:onProperty ofl:OFLID13297 ;   # Ofl part-of
       owl:someValuesFrom ?flap .
    ?flap rdfs:label ?flapLabel .
  }
} ORDER BY ?flapLabel ?pedicleLabel
""", title="CQ3 TBox — named pedicle classes and their parent flap class", graph=g_tbox)

In [ ]:
run("""
SELECT ?flapLabel ?pedicleLabel ?vesselLabel
WHERE {
  ?flap rdf:type owl:NamedIndividual ;
        ofl:OFLID13296 ?pedicle .
  ?pedicle rdf:type ?pedCls .
  ?pedCls rdfs:subClassOf* ofl:OFLID106008 .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?pedicle rdfs:label ?pedicleLabel }
  OPTIONAL {
    ?pedicle ofl:OFLID13296 ?vessel .
    ?vessel rdfs:label ?vesselLabel .
  }
} ORDER BY ?flapLabel LIMIT 10
""", title="CQ3 ABox — flap individuals with their pedicle part and vessels")

### CQ4 — Mathes and Nahai Classification
*What is the Mathes and Nahai classification of the muscle flap?*

**Fulfilled: Yes**

In [ ]:
run("""
SELECT DISTINCT ?flapLabel ?originLabel
WHERE {
  ?flap rdfs:subClassOf* ofl:OFLID10002 .
  ?flap rdfs:subClassOf ?r .
  ?r a owl:Restriction ; owl:onProperty ofl:OFLID12003 ; owl:someValuesFrom ?filler .
  ?flap rdfs:label ?flapLabel .
  {
    FILTER(isIRI(?filler))
    OPTIONAL { ?filler rdfs:label ?originLabel }
  } UNION {
    ?filler owl:unionOf ?lst .
    ?lst rdf:rest*/rdf:first ?member .
    FILTER(isIRI(?member))
    OPTIONAL { ?member rdfs:label ?originLabel }
  }
} ORDER BY ?flapLabel
""", title="CQ4 TBox — flap classes with declared anatomical origin (labels resolved)")

In [ ]:
run("""
SELECT ?flapLabel ?originLabel
WHERE {
  ?flap rdf:type owl:NamedIndividual ;
        ofl:OFLID12003 ?origin .
  OPTIONAL { ?flap   rdfs:label ?flapLabel }
  OPTIONAL { ?origin rdfs:label ?originLabel }
} ORDER BY ?flapLabel LIMIT 10
""", title="CQ4 ABox — generated individuals with anatomical origin (sample)")

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10135

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10135 . FILTER(?t != ofl:OFLID10135)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ4 TBox — all Mathes-Nahai muscle flap types")

In [ ]:
run("""
SELECT ?typeLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10135 . FILTER(?t != ofl:OFLID10135)
  OPTIONAL { ?t rdfs:label ?typeLabel }
} GROUP BY ?typeLabel ORDER BY DESC(?n)
""", title="CQ4 ABox — generated individuals by Mathes-Nahai type")

### CQ5 — Nakajima Classification
*What is the Nakajima²⁴ classification of the flap?*

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10075

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10078

In [ ]:
run("""
SELECT DISTINCT ?nakajimaLabel ?flapLabel
WHERE {
  VALUES ?nakajimaCls {
    ofl:OFLID10075  # Branch-based flap with recognized perforator
    ofl:OFLID10076  # Branch-based flaps
    ofl:OFLID10078  # Perforator-based flaps
  }
  ?flap rdfs:subClassOf* ?nakajimaCls .
  FILTER(?flap != ?nakajimaCls)
  OPTIONAL { ?nakajimaCls rdfs:label ?nakajimaLabel }
  OPTIONAL { ?flap rdfs:label ?flapLabel }
} ORDER BY ?nakajimaLabel ?flapLabel
""", title="CQ5 TBox — Nakajima flap types (branch-based / perforator)", graph=g_tbox)

run("""
SELECT ?nakajimaLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  VALUES ?nakajimaCls {
    ofl:OFLID10075
    ofl:OFLID10076
    ofl:OFLID10078
  }
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ?nakajimaCls .
  OPTIONAL { ?nakajimaCls rdfs:label ?nakajimaLabel }
} GROUP BY ?nakajimaLabel ORDER BY DESC(?n)
""", title="CQ5 ABox — flap individuals by Nakajima classification")

### CQ6 — Transfer Distance
*Is the flap local, regional or distant?*

**Fulfilled: Yes**

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10086

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10086 . FILTER(?t != ofl:OFLID10086)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ6 TBox — distance classification types")

In [ ]:
run("""
SELECT ?distLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10086 . FILTER(?t != ofl:OFLID10086)
  OPTIONAL { ?t rdfs:label ?distLabel }
} GROUP BY ?distLabel ORDER BY DESC(?n)
""", title="CQ6 ABox — flap individuals by distance classification")

### CQ7 — Transfer Method
*Is the flap pedicled or free?*

**Fulfilled: Yes**

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10054
'Flaps without tissue connection to origin'

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10053
'Flaps with tissue connection to origin'

In [ ]:
run("""
SELECT ?transferLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap obo:RO_0000056 ?process .
  ?process rdf:type ?procType .
  ?procType rdfs:subClassOf* ?transferRoot .
  VALUES ?transferRoot { ofl:OFLID1000137 ofl:OFLID1000138 }
  OPTIONAL { ?transferRoot rdfs:label ?transferLabel }
} GROUP BY ?transferLabel ?transferRoot ORDER BY DESC(?n)
""", title="CQ7 ABox — free vs pedicled flap individuals (subClassOf* catches movement subclasses)")

### CQ8 — Vascular Pattern
*Is it a random pattern or an axial or perforator flap?*

**Fulfilled: Yes (TBox)** — random pattern (`OFLID10004`) and non-random pattern (`OFLID10077`) branches are axiomatised with their subclasses. ABox classification depends on reasoning over blood-supply axioms.

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10077 
'Non-random pattern flaps'

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10004
'Random pattern flaps'

In [ ]:
run("""
SELECT DISTINCT ?bloodSupplyLabel ?flapLabel
WHERE {
  VALUES ?bloodSupplyCls {
    ofl:OFLID10004  # Random pattern flaps
    ofl:OFLID10077  # Non-random pattern flaps
    ofl:OFLID10076  # Branch-based flaps
    ofl:OFLID10075  # Branch-based flap with recognized perforator
    ofl:OFLID10078  # Perforator-based flaps
  }
  ?flap rdfs:subClassOf* ?bloodSupplyCls .
  FILTER(?flap != ?bloodSupplyCls)
  OPTIONAL { ?bloodSupplyCls rdfs:label ?bloodSupplyLabel }
  OPTIONAL { ?flap rdfs:label ?flapLabel }
} ORDER BY ?bloodSupplyLabel ?flapLabel
""", title="CQ8 TBox — vascular pattern classification (random / axial / perforator)", graph=g_tbox)

run("""
SELECT ?bloodSupplyLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  VALUES ?bloodSupplyCls {
    ofl:OFLID10004
    ofl:OFLID10077
    ofl:OFLID10076
    ofl:OFLID10075
    ofl:OFLID10078
  }
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ?bloodSupplyCls .
  OPTIONAL { ?bloodSupplyCls rdfs:label ?bloodSupplyLabel }
} GROUP BY ?bloodSupplyLabel ORDER BY DESC(?n)
""", title="CQ8 ABox — flap individuals by vascular pattern")

### CQ9 — Chimeric Flap
*Is it a chimeric flap?*

**Fulfilled: Yes** — `ofl:OFLID1000093` "Flaps classified by chimeric type" is modelled with four subtypes from the Kim 2015 system: Type i (classical), Type ii (anastomotic), Type iii (perforator), Type iv (mixed chimerism).

Targeted class: 
https://purl.bioontology.org/ontology/OFL/OFLID1000092
'Chimeric flaps'

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID1000093 . FILTER(?t != ofl:OFLID1000093)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ9 TBox — chimeric flap subtypes (Kim 2015 classification)", graph=g_tbox)

In [ ]:
run("""
SELECT ?chimericTypeLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID1000093 . FILTER(?t != ofl:OFLID1000093)
  OPTIONAL { ?t rdfs:label ?chimericTypeLabel }
} GROUP BY ?chimericTypeLabel ORDER BY DESC(?n)
""", title="CQ9 ABox — flap individuals by chimeric type")

### CQ10 — Arterial Flow Direction
*What is the arterial flow direction (e.g. reversed flow or flow-through flap)?*

**Fulfilled: Yes (TBox)** — anterograde (`OFLID10113`) and retrograde (`OFLID10123`) flow classes are present under `OFLID10093`. ABox individuals are not currently typed with flow direction.

https://purl.bioontology.org/ontology/OFL/OFLID10093
'Flaps classified by arterial flow direction'

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10093 .
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ10 TBox — arterial flow direction subclass hierarchy", graph=g_tbox)

run("""
SELECT ?flowLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  VALUES ?flowCls {
    ofl:OFLID10113  # Flaps with anterograde blood flow
    ofl:OFLID10123  # Flaps with retrograde blood flow
  }
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ?flowCls .
  OPTIONAL { ?flowCls rdfs:label ?flowLabel }
} GROUP BY ?flowLabel ORDER BY DESC(?n)
""", title="CQ10 ABox — flap individuals by arterial flow direction")

### CQ11 — Movement Type
*Is it a rotational, advancement or transpositional flap?*

**Fulfilled: Yes (TBox + ABox)**

Movement classification is inference-driven via three defined classes:

| Class | IRI | equivalentClass restriction |
|-------|-----|-----------------------------|
| Rotation flaps | `OFLID10014` | `OFLID10068 AND (participates_in some OFLID120025)` |
| Transposition flaps | `OFLID10067` | `OFLID10068 AND (participates_in some OFLID120057)` |
| Advancement flaps | `OFLID10070` | `OFLID10068 AND (participates_in some OFLID120056)` |

`OFLID10068` ("Flaps classified by movement") itself requires the flap to `participates_in` a complete operation that `has_part` a pedicled transfer — so **only pedicled flaps** are classified by movement type.

In the ABox, each pedicled flap's transfer process individual is typed with the specific movement subclass (`OFLID120025 / OFLID120056 / OFLID120057`) rather than the generic `PEDICLED_TRANSFER (OFLID1000138)`. All three are `subClassOf OFLID1000138`, so OWL RL infers `PEDICLED_TRANSFER` upward while also satisfying the `participates_in` restriction that fires `OFLID10014 / OFLID10067 / OFLID10070`.

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10068 . FILTER(?t != ofl:OFLID10068)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ11 TBox — movement-type flap subclasses", graph=g_tbox)

In [ ]:
run("""
SELECT ?movementLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ?movement .
  VALUES ?movement { ofl:OFLID10014 ofl:OFLID10067 ofl:OFLID10070 ofl:OFLID10132 }
  OPTIONAL { ?movement rdfs:label ?movementLabel }
} GROUP BY ?movementLabel ORDER BY DESC(?n)
""", title="CQ11 ABox — flap individuals by movement type (rotation / transposition / advancement / interpolation)")

### CQ12 — Island Flap
*Is it an island flap?*

**Fulfilled: Yes (TBox)** — `OFLID10115` (Cutaneous island flap) and its subclasses are present. ABox individuals are not currently typed with island flap classification.

Targeted class: https://purl.bioontology.org/ontology/OFL/OFLID10115
'Cutaneous island flap'

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10115 .
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ12 TBox — island flap subclass hierarchy", graph=g_tbox)

run("""
SELECT ?islandLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10115 .
  OPTIONAL { ?t rdfs:label ?islandLabel }
} GROUP BY ?islandLabel ORDER BY DESC(?n)
""", title="CQ12 ABox — island flap individuals by subtype")

### CQ13 — Insertion Site Preparation
*How was the insertion site prepared?*

**Fulfilled: Partially** — the inset process class hierarchy is present in the TBox; ABox individuals for inset processes are not yet generated.

https://purl.bioontology.org/ontology/OFL/OFLID130001
'Flaps classified by insertion site preparation'

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID1000141 .
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ13 TBox — insertion process subclass hierarchy (partially fulfilled)")

### CQ14 — Pre-Harvest Modification
*Was the flap modified before harvesting?*

**Fulfilled: Partially** — prefabrication and prelamination classes are present; full axiomatisation of the modification procedures themselves is ongoing.

https://purl.bioontology.org/ontology/OFL/OFLID10045
'Flaps classified by pre harvest modification'

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10131 . FILTER(?t != ofl:OFLID10131)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ8 TBox — pre-harvest modification types")

In [ ]:
run("""
SELECT ?modLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10045 . FILTER(?t != ofl:OFLID10045)
  OPTIONAL { ?t rdfs:label ?modLabel }
} GROUP BY ?modLabel ORDER BY DESC(?n)
""", title="CQ8 ABox — pre-harvest modification counts")

### CQ15 — Split-Thickness Skin Grafting
*Did the flap require split-thickness skin grafting?*

**Fulfilled: No** — the STSG concept and its bibliographic citation are present in the TBox, but the relationship between STSG and recipient flaps is not yet axiomatised.

https://purl.bioontology.org/ontology/OFL/OFLID120066
'Flaps with skin graft'

In [ ]:
run("""
SELECT ?label ?citation
WHERE {
  OPTIONAL { ofl:OFLID1000185 rdfs:label ?label }
  OPTIONAL { ofl:OFLID1000185 dcterms:bibliographicCitation ?citation }
}
""", title="CQ9 — STSG concept in TBox (not yet linked to donor-site flaps)")

### CQ16 — Vessel Anastomosis
*How was vessel anastomosis performed?*

https://purl.bioontology.org/ontology/OFL/OFLID10223
'Flaps classified by configuration of vessel anastomosis'

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID1000171 . FILTER(?t != ofl:OFLID1000171)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ16 TBox — anastomosis classified by vessel type (arterial / venous / AV)", graph=g_tbox)

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID1000175 . FILTER(?t != ofl:OFLID1000175)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ16 TBox — anastomosis classified by configuration (end-to-end / end-to-side)", graph=g_tbox)

In [ ]:
run("""
SELECT ?anastomosisTypeLabel (COUNT(DISTINCT ?anast) AS ?n)
WHERE {
  ?anast rdf:type ?t .
  ?t rdfs:subClassOf* ?root .
  VALUES ?root { ofl:OFLID1000170 ofl:OFLID1000175 }
  FILTER(?t != ?root)
  OPTIONAL { ?t rdfs:label ?anastomosisTypeLabel }
} GROUP BY ?anastomosisTypeLabel ORDER BY DESC(?n)
""", title="CQ16 ABox — vessel anastomosis individuals by type (if generated)")

### CQ17 — Flap Survival
*Did the flap fully survive?*

**Fulfilled: Yes**

https://purl.bioontology.org/ontology/OFL/OFLID10176
'Flaps with tissue loss'

https://purl.bioontology.org/ontology/OFL/OFLID10184
'Flaps without tissue loss'

In [ ]:
run("""
SELECT DISTINCT ?label
WHERE {
  VALUES ?survRoot { ofl:OFLID10184 ofl:OFLID10176 }
  ?t rdfs:subClassOf* ?survRoot .
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ17 TBox — survival outcome classes (OFLID10184 no-loss; OFLID10176 tissue-loss branch)")

In [ ]:
run("""
SELECT ?outcomeLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?survCls rdfs:label ?outcomeLabel }
} GROUP BY ?outcomeLabel ORDER BY DESC(?n)
""", title="CQ17 ABox — survival outcome counts")

### Summary Statistics

In [ ]:
print("\n" + "="*60)
print("Grand totals")
n_flaps = list(g.query(PREFIXES + "SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?f rdf:type ofl:OFLID10002 . }"))[0][0]
n_ind   = sum(1 for _ in g.subjects(RDF.type, OWL.NamedIndividual))
print(f"  Flap individuals (direct type ofl:OFLID10002) : {n_flaps}")
print(f"  All named individuals                         : {n_ind:,}")
print(f"  Total triples (TBox + ABox + FMA)             : {len(g):,}")

---
## Task-Based Evaluation

Five realistic clinical and research tasks demonstrating the ontology supports end-to-end use cases beyond abstract CQ answering.

### Task 1 — Flap selection by donor region
**Clinical scenario:** A surgeon needs a pedicled flap for trunk reconstruction and wants to know which flap types originate from the back of the trunk and are transferred regionally.

*Query: find all flap classes whose `has-flap-origin` restriction points to "Back of trunk", then count how many ABox individuals of those classes are pedicled.*

In [ ]:
# Task 1a — TBox: which flap classes originate from the back of the trunk?
run("""
SELECT DISTINCT ?flapLabel ?originLabel
WHERE {
  ?flap rdfs:subClassOf* ofl:OFLID10002 .
  ?flap rdfs:subClassOf ?r .
  ?r a owl:Restriction ; owl:onProperty ofl:OFLID12003 ; owl:someValuesFrom ?origin .
  ?origin rdfs:label ?originLabel .
  FILTER(CONTAINS(LCASE(?originLabel), "back of trunk"))
  ?flap rdfs:label ?flapLabel .
} ORDER BY ?flapLabel
""", title="Task 1a — Flap classes with origin in 'Back of trunk'")

In [ ]:
# Task 1b — ABox: pedicled instances of those flap types with their survival
run("""
SELECT ?flapLabel ?survivalLabel
WHERE {
  ?flap ofl:OFLID12003 ?origin .
  ?origin rdfs:label ?originLabel .
  FILTER(CONTAINS(LCASE(?originLabel), "back of trunk"))
  ?flap obo:RO_0000056 ?process .
  ?process rdf:type ?procType .
  ?procType rdfs:subClassOf* ofl:OFLID1000138 .
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  ?flap rdfs:label ?flapLabel .
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} ORDER BY ?flapLabel LIMIT 15
""", title="Task 1b — Pedicled back-of-trunk flap individuals with survival outcome (sample)")

### Task 2 — Free flap survival audit
**Research scenario:** A clinical researcher wants to audit all free flap procedures and produce a survival distribution table — the kind of result reported in outcome studies.

*Query: for every free flap individual, retrieve its survival classification and aggregate counts.*

In [ ]:
run("""
SELECT ?survivalLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap obo:RO_0000056 ?process .   # participates in
  ?process rdf:type ofl:OFLID1000137 .     # free transfer
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} GROUP BY ?survivalLabel ORDER BY DESC(?n)
""", title="Task 2 — Survival distribution across all free flap individuals")

### Task 3 — Mathes-Nahai Type I flap lookup
**Educational scenario:** A resident wants to know which muscle flaps have a single dominant vascular pedicle (Mathes-Nahai Type I) and how many cases of each are represented.

*Query: retrieve all flap classes that are subclasses of Type I, and count ABox individuals per type.*

In [ ]:
# TBox: for each Mathes-Nahai type, which named flap classes are subclasses of it?
run("""
SELECT DISTINCT ?mnTypeLabel ?flapLabel
WHERE {
  ?mnType rdfs:subClassOf ofl:OFLID10135 .    # direct children of Mathes-Nahai root
  ?flap rdfs:subClassOf* ?mnType .
  FILTER(?flap != ?mnType)
  FILTER(STRSTARTS(STR(?flap), "https://purl.bioontology.org/ontology/OFL/"))
  ?mnType rdfs:label ?mnTypeLabel .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
} ORDER BY ?mnTypeLabel ?flapLabel
""", title="Task 3a — Named flap classes per Mathes-Nahai vascular pattern type")

In [ ]:
# ABox: count instances per Mathes-Nahai type (flap individuals typed to subclasses of OFLID10135)
run("""
SELECT ?mnTypeLabel ?flapTypeLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?flapType .
  ?flapType rdfs:subClassOf* ?mnType .
  ?mnType rdfs:subClassOf ofl:OFLID10135 .
  OPTIONAL { ?mnType rdfs:label ?mnTypeLabel }
  OPTIONAL { ?flapType rdfs:label ?flapTypeLabel }
} GROUP BY ?mnTypeLabel ?flapTypeLabel ORDER BY ?mnTypeLabel DESC(?n)
""", title="Task 3b — ABox instance counts per Mathes-Nahai type and flap class")

### Task 4 — Prefabrication outcome tracking
**Research scenario:** A researcher investigates whether pre-harvest modification affects flap survival. Query all prefabricated flap individuals and cross-tabulate with survival outcome.

In [ ]:
run("""
SELECT ?survivalLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ofl:OFLID10131 .          # flaps with preharvest modification
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} GROUP BY ?survivalLabel ORDER BY DESC(?n)
""", title="Task 4 — Survival outcomes for pre-harvest modified flaps")

### Task 5 — Operative planning: anatomical component inventory
**Clinical scenario:** Before harvesting a Fibula flap, the surgeon wants a complete list of anatomical structures defined in the ontology as components of this flap — directly queryable from the TBox without needing a textbook.

In [ ]:
# TBox: has-part restrictions on Fibula flap — resolve union/intersection fillers via ABox component types
# Since Fibula flap fillers are union blank nodes, we query the ABox component individuals
# and retrieve their FMA class labels (same information, grounded in instances)
run("""
SELECT DISTINCT ?typeLabel (COUNT(DISTINCT ?component) AS ?n)
WHERE {
  ?flap rdfs:label ?fl . FILTER(CONTAINS(LCASE(?fl), "fibula flap"))
  ?flap ofl:OFLID13296 ?component .
  ?component rdf:type ?t .
  FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
  OPTIONAL { ?t rdfs:label ?typeLabel }
} GROUP BY ?typeLabel ORDER BY ?typeLabel
""", title="Task 5 — Distinct FMA anatomical types harvested in Fibula flap instances")

In [ ]:
# ABox: actual component individuals of a generated Fibula flap instance
run("""
SELECT ?componentLabel ?typeLabel
WHERE {
  ?flap rdfs:label ?fl . FILTER(CONTAINS(LCASE(?fl), "fibula flap"))
  ?flap ofl:OFLID13296 ?component .
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
} ORDER BY ?componentLabel LIMIT 12
""", title="Task 5 — Component individuals of a generated Fibula flap ABox instance")